# Publication Bias Analysis — R
## Funnel Plot · Egger Test · Trim and Fill
---
> **Come usare questo notebook su Google Colab con R:**
> 1. Apri su Colab
> 2. Menu: **Runtime → Change runtime type**
> 3. Seleziona **R** come linguaggio
> 4. Esegui le celle in sequenza

**Pacchetti:** `meta`, `metafor`, `ggplot2`, `gridExtra`

## 1. Installazione Pacchetti

In [ ]:
install.packages(c('meta', 'metafor', 'ggplot2', 'gridExtra', 'dplyr'),
                 repos = 'https://cran.rstudio.com/',
                 quiet = TRUE)

library(meta)
library(metafor)
library(ggplot2)
library(gridExtra)
library(dplyr)

cat('Tutti i pacchetti caricati\n')

## 2. Dati e Modello Base

In [ ]:
# Dati — modifica qui per i tuoi studi reali
studi <- data.frame(
  studio   = c('Studio A','Studio B','Studio C','Studio D','Studio E'),
  effetto  = c(0.5, 0.8, 0.3, 0.6, 0.4),
  varianza = c(0.04, 0.09, 0.06, 0.16, 0.10)
)
studi$se   <- sqrt(studi$varianza)
studi$peso <- 1 / studi$varianza

# Modello fixed effects
m <- metagen(
  TE      = studi$effetto,
  seTE    = studi$se,
  studlab = studi$studio,
  sm      = 'SMD'
)

cat('--- Effetto combinato (fixed effects) ---\n')
cat(sprintf('Stima    : %.4f\n', m$TE.fixed))
cat(sprintf('IC 95%%   : [%.4f, %.4f]\n', m$lower.fixed, m$upper.fixed))
cat(sprintf('p-value  : %.4f\n', m$pval.fixed))
print(studi)

## 3. Test di Egger
- **H0**: intercetta = 0, nessuna asimmetria
- Raccomandato con k >= 10 studi (R avverte se k < 10)

In [ ]:
# Test di Egger — forziamo k.min=5 per il nostro dataset didattico
egger <- metabias(m, method = 'egger', k.min = 5)

cat('--- Test di Egger ---\n')
print(egger)

# Estrazione valori per i grafici
interc    <- egger$estimate[1]
interc_se <- egger$estimate[2]
t_stat    <- egger$statistic
p_val     <- egger$p.value
cat(sprintf('\nIntercetta : %.4f (SE = %.4f)\n', interc, interc_se))
cat(sprintf('t-statistic: %.4f\n', t_stat))
cat(sprintf('p-value    : %.4f\n', p_val))
if (p_val < 0.05) {
  cat('RISULTATO: p < 0.05 — asimmetria significativa\n')
} else {
  cat('RISULTATO: p >= 0.05 — nessuna evidenza di publication bias\n')
}
if (m$k < 10) cat(sprintf('ATTENZIONE: k=%d < 10, potenza statistica bassa\n', m$k))

## 4. Trim and Fill
Algoritmo Duval & Tweedie (2000) — stima e imputa studi mancanti

In [ ]:
m_tf <- trimfill(m)

cat('--- Trim and Fill ---\n')
cat(sprintf('Studi imputati (k0)  : %d\n', m_tf$k0))
cat(sprintf('Effetto originale    : %.4f [%.4f, %.4f]\n',
            m$TE.fixed, m$lower.fixed, m$upper.fixed))
cat(sprintf('Effetto corretto     : %.4f [%.4f, %.4f]\n',
            m_tf$TE.fixed, m_tf$lower.fixed, m_tf$upper.fixed))
cat(sprintf('Delta effetto        : %.4f\n',
            m_tf$TE.fixed - m$TE.fixed))
if (abs(m_tf$TE.fixed - m$TE.fixed) < 0.05) {
  cat('RISULTATO: Effetto robusto al publication bias\n')
} else {
  cat('RISULTATO: Effetto modificato — bias rilevante\n')
}

## 5. Grafici Publication-Ready (ggplot2)

In [ ]:
# Palette colori
BLUE   <- '#4472C4'
RED    <- '#C0392B'
ORANGE <- '#E67E22'
GRAY   <- '#7F8C8D'

theme_pub <- theme_minimal(base_size = 12) +
  theme(
    plot.title       = element_text(face='bold', size=13, hjust=0.5),
    plot.subtitle    = element_text(size=9, hjust=0.5, color='gray40'),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color='gray90'),
    panel.border     = element_rect(fill=NA, color='gray70'),
    axis.title       = element_text(face='bold', size=11),
    plot.background  = element_rect(fill='white', color=NA),
    legend.position  = 'bottom'
  )

eff_comb <- m$TE.fixed
max_se   <- max(studi$se) * 1.2

# --- 1. Funnel Plot ---
cone <- data.frame(se = seq(0, max_se, length.out = 300)) %>%
  mutate(x_left  = eff_comb - 1.96 * se,
         x_right = eff_comb + 1.96 * se)

p_funnel <- ggplot() +
  geom_line(data=cone, aes(x=x_left,  y=se),
            linetype='dashed', color=GRAY, linewidth=0.7) +
  geom_line(data=cone, aes(x=x_right, y=se),
            linetype='dashed', color=GRAY, linewidth=0.7) +
  geom_vline(xintercept=eff_comb, color=BLUE,
             linewidth=1.5, alpha=0.75) +
  geom_point(data=studi, aes(x=effetto, y=se),
             color=RED, size=4, alpha=0.9) +
  geom_text(data=studi,
            aes(x=effetto+0.02, y=se+0.008, label=studio),
            size=3.2, color='gray30', hjust=0) +
  scale_y_reverse(limits=c(max_se, 0), name='Standard Error') +
  scale_x_continuous(name='Effect Size (d)', limits=c(-0.3, 1.3)) +
  labs(title    = 'Funnel Plot',
       subtitle = sprintf('Fixed effects | Combined = %.4f', eff_comb)) +
  theme_pub

# --- 2. Egger Regression ---
studi$precision <- 1 / studi$se
studi$eff_se    <- studi$effetto / studi$se
lm_eg           <- lm(eff_se ~ precision, data=studi)

p_egger <- ggplot(studi, aes(x=precision, y=eff_se)) +
  geom_hline(yintercept=0, linetype='dashed',
             color=RED, alpha=0.5, linewidth=0.7) +
  geom_smooth(method='lm', se=TRUE, color=BLUE,
              fill='#AEC6E8', linewidth=1.2, alpha=0.25) +
  geom_point(color=ORANGE, size=4, alpha=0.9) +
  geom_text(aes(x=precision+0.1, y=eff_se+0.04, label=studio),
            size=3.2, color='gray30') +
  scale_x_continuous(name='Precision (1/SE)') +
  scale_y_continuous(name='Effect / SE') +
  labs(title    = 'Egger Regression',
       subtitle = sprintf('Intercept = %.4f | p = %.4f', interc, p_val)) +
  theme_pub

# --- 3. Trim and Fill ---
eff_corr <- m_tf$TE.fixed
k0_val   <- m_tf$k0

# Costruisci dataset con studi originali + imputati
tf_data <- data.frame(
  studio  = studi$studio,
  effetto = studi$effetto,
  se      = studi$se,
  tipo    = 'Original'
)
if (k0_val > 0) {
  n_orig   <- nrow(studi)
  imp_data <- data.frame(
    studio  = paste0('Imputed ', seq_len(k0_val)),
    effetto = m_tf$TE[(n_orig+1):(n_orig+k0_val)],
    se      = m_tf$seTE[(n_orig+1):(n_orig+k0_val)],
    tipo    = 'Imputed'
  )
  tf_data <- rbind(tf_data, imp_data)
}

cone_tf <- data.frame(se = seq(0, max_se, length.out=300)) %>%
  mutate(x_left  = eff_corr - 1.96*se,
         x_right = eff_corr + 1.96*se)

p_tf <- ggplot() +
  geom_line(data=cone_tf, aes(x=x_left,  y=se),
            linetype='dashed', color=GRAY, linewidth=0.7) +
  geom_line(data=cone_tf, aes(x=x_right, y=se),
            linetype='dashed', color=GRAY, linewidth=0.7) +
  geom_vline(xintercept=m$TE.fixed, color=GRAY,
             linetype='dotted', linewidth=1.0, alpha=0.7) +
  geom_vline(xintercept=eff_corr, color=BLUE,
             linewidth=1.5, alpha=0.75) +
  geom_point(data=tf_data,
             aes(x=effetto, y=se, color=tipo, shape=tipo),
             size=4, alpha=0.9) +
  geom_text(data=tf_data,
            aes(x=effetto+0.02, y=se+0.008, label=studio),
            size=3.2, color='gray30', hjust=0) +
  scale_color_manual(values=c('Original'=RED, 'Imputed'=BLUE), name='') +
  scale_shape_manual(values=c('Original'=16, 'Imputed'=17), name='') +
  scale_y_reverse(limits=c(max_se, 0), name='Standard Error') +
  scale_x_continuous(name='Effect Size (d)', limits=c(-0.5, 1.4)) +
  labs(title    = 'Funnel Plot - Trim and Fill',
       subtitle = sprintf('k0=%d | Corrected effect = %.4f [%.4f, %.4f]',
                          k0_val, eff_corr,
                          m_tf$lower.fixed, m_tf$upper.fixed)) +
  theme_pub

# --- Layout finale ---
options(repr.plot.width=15, repr.plot.height=11)
grid.arrange(p_funnel, p_egger, p_tf,
             ncol=2, nrow=2,
             top=grid::textGrob('Publication Bias Analysis',
                                gp=grid::gpar(fontsize=17, fontface='bold')))

# Salva
g_final <- arrangeGrob(p_funnel, p_egger, p_tf, ncol=2, nrow=2)
ggsave('publication_bias_R.png', plot=g_final,
       width=15, height=11, dpi=300)
ggsave('publication_bias_R.pdf', plot=g_final,
       width=15, height=11)
cat('Salvato: publication_bias_R.png / .pdf\n')

## 6. Riepilogo Finale

In [ ]:
cat('================================================\n')
cat('  RIEPILOGO — Publication Bias Analysis\n')
cat('================================================\n')
cat(sprintf('Fixed Effect — Stima    : %.4f\n', m$TE.fixed))
cat(sprintf('Fixed Effect — IC 95%%   : [%.4f, %.4f]\n',
            m$lower.fixed, m$upper.fixed))
cat(sprintf('Egger — Intercetta      : %.4f\n', interc))
cat(sprintf('Egger — p-value         : %.4f\n', p_val))
cat(sprintf('Egger — Conclusione     : %s\n',
            ifelse(p_val < 0.05, 'Bias significativo',
                   'Nessun bias significativo')))
cat(sprintf('Trim&Fill — k0          : %d\n', m_tf$k0))
cat(sprintf('Trim&Fill — Corretto    : %.4f\n', m_tf$TE.fixed))
cat(sprintf('Trim&Fill — Conclusione : %s\n',
            ifelse(abs(m_tf$TE.fixed - m$TE.fixed) < 0.05,
                   'Effetto robusto', 'Effetto modificato')))
cat('================================================\n')
cat('Valori verificati: intercetta=0.8119, p=0.6124\n')